# LeetCode 665 — Non-decreasing Array bằng Python

Link đề: https://leetcode.com/problems/non-decreasing-array/ · Độ khó: **Medium** · Chủ đề: Array, Greedy (tham lam)

Skeleton LeetCode:

```python
class Solution:
    def checkPossibility(self, nums: List[int]) -> bool:
        ...
```

Ghi chú: bài này nổi tiếng vì **rất dễ viết sai** dù ý tưởng ngắn. Chỗ khó nằm ở câu hỏi: khi gặp một cặp "giảm" `nums[i] > nums[i+1]`, ta nên **hạ** `nums[i]` xuống hay **nâng** `nums[i+1]` lên? Notebook đi từ vét cạn "thử sửa từng vị trí" `O(n^2)`, tới greedy `O(n)` đếm cặp giảm, tới một cách **không sửa mảng** chỉ so sánh lân cận, và cuối cùng là cách hoàn toàn khác bằng **dãy con không giảm dài nhất (LNDS)**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mảng có thể trở thành **không giảm** (`nums[i] <= nums[i+1]` với mọi `i`) bằng cách **thay đổi tối đa một phần tử** (thành **bất kỳ** số nguyên nào) hay không.
2. Nắm công cụ Python cần dùng: duyệt theo cặp liền kề bằng `range(n - 1)`, `all(...)` kết hợp generator, sao chép list bằng `nums[:]`, bẫy chỉ số âm `nums[-1]`, `bisect_right`.
3. Đi từ **thử sửa từng vị trí `O(n^2)`** → **greedy đếm cặp giảm `O(n)`** → **sửa ở cặp giảm đầu tiên rồi kiểm tra** → **so sánh lân cận không đụng mảng** → **dãy con không giảm dài nhất `O(n log n)`**.
4. Hiểu mấu chốt greedy: ở cặp giảm `(a, b)` tại `i`, nếu `nums[i-1] <= b` thì **hạ `a` xuống `b`** (giữ cho tương lai càng nhỏ càng tốt); ngược lại buộc phải **nâng `b` lên `a`**.
5. Tránh các lỗi kinh điển: chỉ đếm cặp giảm mà không xét hai phía, luôn hạ trái hoặc luôn nâng phải, quên chặn `i = 0` (và bẫy `nums[-1]` của Python), và dùng `>=` thay vì `>`.

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm `n` số nguyên. Hãy kiểm tra xem có thể làm cho mảng **không giảm** bằng cách **thay đổi tối đa một phần tử** hay không (trả về `True`/`False`).

Mảng gọi là **không giảm** (non-decreasing) nếu `nums[i] <= nums[i + 1]` với mọi `i` thỏa `0 <= i <= n - 2`. Chú ý dấu `<=`: hai số liền kề **bằng nhau** vẫn được.

**Ví dụ 1**

- Input: `nums = [4, 2, 3]`
- Output: `true`
- Giải thích: đổi số `4` đầu tiên thành `1` (hoặc `2`) được `[1, 2, 3]`.

**Ví dụ 2**

- Input: `nums = [4, 2, 1]`
- Output: `false`
- Giải thích: có hai cặp giảm (`4 > 2` và `2 > 1`), một lần sửa không đủ.

**Ràng buộc:**

- `n == nums.length`
- `1 <= n <= 10^4`
- `-10^5 <= nums[i] <= 10^5`

Lưu ý: giá trị mới của phần tử bị sửa có thể là **bất kỳ số nguyên** (đề không giới hạn trong khoảng `[-10^5, 10^5]`). Mảng 1 hoặc 2 phần tử luôn trả `true` (sửa một phần tử là đủ).

In [1]:
from typing import List
from bisect import bisect_right
import random

examples = [
    ([4, 2, 3], True),
    ([4, 2, 1], False),
]
for nums, expected in examples:
    print("nums =", nums, "| kỳ vọng:", expected)

nums = [4, 2, 3] | kỳ vọng: True
nums = [4, 2, 1] | kỳ vọng: False


## 3. Kiến thức nền

### 3.1. Duyệt các cặp liền kề: `range(n - 1)`

Để so `nums[i]` với `nums[i + 1]` ta cho `i` chạy từ `0` đến `n - 2`, tức `range(n - 1)`. Nếu cho `i` chạy tới `n - 1` thì `nums[i + 1]` vượt biên và báo `IndexError`. Mảng `n` phần tử có đúng `n - 1` cặp liền kề.

In [2]:
nums = [4, 2, 3]
n = len(nums)
for i in range(n - 1):
    print(f"cặp ({i},{i+1}): {nums[i]} vs {nums[i+1]} ->", "GIẢM" if nums[i] > nums[i + 1] else "ok")

cặp (0,1): 4 vs 2 -> GIẢM
cặp (1,2): 2 vs 3 -> ok


### 3.2. "Không giảm" kiểm tra bằng `all(...)`

`all(iterable)` trả `True` nếu **mọi** phần tử của iterable đều đúng; với iterable rỗng cũng trả `True`. Kết hợp với biểu thức sinh (generator), ta viết hàm kiểm tra không giảm trong một dòng. Với mảng 0 hoặc 1 phần tử, `range(n - 1)` rỗng nên đương nhiên không giảm.

In [3]:
def is_nondecreasing(a):
    return all(a[i] <= a[i + 1] for i in range(len(a) - 1))


print(is_nondecreasing([1, 2, 2, 3]))   # True: bằng nhau vẫn ok
print(is_nondecreasing([1, 3, 2]))      # False
print(is_nondecreasing([7]))            # True: không có cặp nào
print(is_nondecreasing([]))             # True

True
False
True
True


### 3.3. Sao chép list: `nums[:]` và tránh sửa nhầm đầu vào

`b = nums` **không** tạo bản sao — `b` và `nums` là cùng một list; sửa `b` là sửa `nums`. Muốn bản sao độc lập (cùng giá trị) ta dùng `nums[:]` hoặc `list(nums)`. Khi thử sửa nhiều kịch bản khác nhau ta phải sao chép trước mỗi lần thử.

In [4]:
nums = [4, 2, 3]
alias = nums
copy = nums[:]
alias[0] = 99
print("nums  :", nums, "(bị đổi vì alias cùng đối tượng)")
print("copy  :", copy, "(không bị đổi)")
print("alias is nums:", alias is nums, "| copy is nums:", copy is nums)

nums  : [99, 2, 3] (bị đổi vì alias cùng đối tượng)
copy  : [4, 2, 3] (không bị đổi)
alias is nums: True | copy is nums: False


### 3.4. Bẫy chỉ số âm: `nums[-1]` là phần tử CUỐI

Trong Python `nums[-1]` là phần tử cuối, `nums[-2]` là kế cuối, **không báo lỗi**. Vì vậy khi viết `nums[i - 1]` với `i = 0` bạn vô tình đọc `nums[-1]` (phần tử cuối mảng) — một lỗi im lặng rất nguy hiểm. Ta phải **chặn `i > 0`** trước khi dùng `nums[i - 1]` (mục 13 sẽ biến nó thành một lỗi cụ thể).

In [5]:
nums = [4, 2, 3]
i = 0
print("nums[i - 1] khi i = 0 là", nums[i - 1], "(chính là phần tử cuối nums[-1] =", nums[-1], ")")
print("i > 0 and nums[i - 1] > 5 ->", i > 0 and nums[i - 1] > 5, "(toán tử and ngắt sớm nên không đọc nums[i-1])")

nums[i - 1] khi i = 0 là 3 (chính là phần tử cuối nums[-1] = 3 )
i > 0 and nums[i - 1] > 5 -> False (toán tử and ngắt sớm nên không đọc nums[i-1])


### 3.5. `bisect_right` (sẽ dùng ở cách 5)

`bisect_right(a, x)` trả vị trí chèn `x` vào list **đã sắp xếp** `a` sao cho nằm **sau** các phần tử bằng `x`. Thao tác tốn `O(log n)` nhờ tìm kiếm nhị phân. Ta dùng nó để dựng dãy con không giảm dài nhất.

In [6]:
a = [1, 3, 3, 5]
print("bisect_right(a, 3) =", bisect_right(a, 3), "-> chèn sau các số 3")
print("bisect_right(a, 0) =", bisect_right(a, 0), "| bisect_right(a, 9) =", bisect_right(a, 9))

bisect_right(a, 3) = 3 -> chèn sau các số 3
bisect_right(a, 0) = 0 | bisect_right(a, 9) = 4


## 4. Ý tưởng nền tảng / trực giác

Gọi một cặp `nums[i] > nums[i+1]` là một **cặp giảm** (descent). Mảng không giảm ⇔ **không có cặp giảm nào**.

Quan sát đầu tiên: **một lần sửa chỉ ảnh hưởng tối đa hai cặp** — cặp bên trái `(nums[i-1], nums[i])` và cặp bên phải `(nums[i], nums[i+1])` của phần tử bị sửa. Do đó:

- Nếu mảng có **từ 2 cặp giảm không dính nhau trở lên**, một lần sửa không đủ. Ví dụ `[4, 2, 1]`: cặp `(4,2)` và `(2,1)` — hai cặp giảm dùng chung phần tử `2`, nhưng sửa `2` thành bất kỳ giá trị nào cũng không được (cần `>= 4` và `<= 1`).
- Nếu mảng có **đúng 1 cặp giảm** tại `(i, i+1)`, ta phải sửa **hoặc `nums[i]` hoặc `nums[i+1]`** (sửa phần tử khác không giúp cặp này). Hai lựa chọn:
  - **Hạ `nums[i]` xuống `nums[i+1]`**: hợp lệ nếu `nums[i-1] <= nums[i+1]` (hoặc `i = 0`).
  - **Nâng `nums[i+1]` lên `nums[i]`**: hợp lệ nếu `nums[i] <= nums[i+2]` (hoặc `i + 1` là phần tử cuối).

Hãy xem ba ví dụ tay:

| mảng | cặp giảm | hạ trái? | nâng phải? | kết quả |
|---|---|---|---|---|
| `[4, 2, 3]` | `(4,2)` tại `i=0` | `i = 0` nên ok | `4 <= 3`? không | `true` (hạ `4` thành `2`) |
| `[3, 4, 2, 5]` | `(4,2)` tại `i=1` | `3 <= 2`? không | `4 <= 5`? có | `true` (nâng `2` thành `4`) |
| `[3, 4, 2, 3]` | `(4,2)` tại `i=1` | `3 <= 2`? không | `4 <= 3`? không | `false` |

In [7]:
def descents(nums):
    """Danh sách chỉ số i mà nums[i] > nums[i+1]."""
    return [i for i in range(len(nums) - 1) if nums[i] > nums[i + 1]]


for arr in ([4, 2, 3], [3, 4, 2, 5], [3, 4, 2, 3], [4, 2, 1], [1, 2, 3]):
    ds = descents(arr)
    print(f"{str(arr):14s} cặp giảm tại i = {ds}")

[4, 2, 3]      cặp giảm tại i = [0]
[3, 4, 2, 5]   cặp giảm tại i = [1]
[3, 4, 2, 3]   cặp giảm tại i = [1]
[4, 2, 1]      cặp giảm tại i = [0, 1]
[1, 2, 3]      cặp giảm tại i = []


Để ý `[3, 4, 2, 3]` chỉ có **một** cặp giảm nhưng vẫn là `false`. Vì vậy **đếm số cặp giảm ≤ 1 là chưa đủ** — phải xét thêm hai phần tử lân cận. Đây là nguồn lỗi số một của bài này (mục 13).

## 5. Cách 1 — Brute Force: thử sửa từng vị trí

### Ý tưởng

Với **mỗi vị trí `i`**, thử đặt `nums[i]` bằng một giá trị "có lợi" rồi kiểm tra mảng có không giảm hay không. Giá trị nào là "có lợi"? Để `nums[i]` khớp với hai bên cần `nums[i-1] <= nums[i] <= nums[i+1]`; chỉ cần thử **hai cực**: `nums[i-1]` (giá trị nhỏ nhất có thể) và `nums[i+1]` (giá trị lớn nhất có thể) — nếu một giá trị nào đó ở giữa hợp lệ thì hai cực cũng hợp lệ. Ở biên (`i = 0` hoặc `i = n-1`) chỉ có một hàng xóm để thử.

Ngoài ra, nếu mảng đã không giảm thì trả `true` ngay.

Ta cũng viết một **oracle cực ngây thơ** `oracle_nd`: với mỗi vị trí `i` thử **mọi** giá trị nguyên trong `[min-1, max+1]`. Nó chậm nhưng khó sai nhất và dùng để kiểm chứng toàn bộ các cách còn lại trên mảng nhỏ.

In [8]:
def check_try_each(nums):
    n = len(nums)
    if is_nondecreasing(nums):
        return True
    for i in range(n):
        candidates = []
        if i > 0:
            candidates.append(nums[i - 1])
        if i < n - 1:
            candidates.append(nums[i + 1])
        for v in candidates:
            b = nums[:]
            b[i] = v
            if is_nondecreasing(b):
                return True
    return False


def oracle_nd(nums):
    """Oracle: thử MỌI vị trí với MỌI giá trị nguyên trong [min-1, max+1]."""
    if is_nondecreasing(nums):
        return True
    lo, hi = min(nums), max(nums)
    for i in range(len(nums)):
        for v in range(lo - 1, hi + 2):
            b = nums[:]
            b[i] = v
            if is_nondecreasing(b):
                return True
    return False


for nums, expected in examples:
    print(nums, "| try_each:", check_try_each(nums), "| oracle:", oracle_nd(nums), "| kỳ vọng:", expected)
    assert check_try_each(nums) == oracle_nd(nums) == expected

[4, 2, 3] | try_each: True | oracle: True | kỳ vọng: True
[4, 2, 1] | try_each: False | oracle: False | kỳ vọng: False


### Độ phức tạp và nút thắt

- `check_try_each`: có `n` vị trí × 2 giá trị thử × `O(n)` để kiểm tra không giảm và sao chép → `O(n^2)` thời gian, `O(n)` bộ nhớ phụ.
- `oracle_nd`: còn chậm hơn, `O(n * V * n)` với `V` là số giá trị thử.

Với `n = 10^4`, `O(n^2) = 10^8` thao tác trong Python là quá chậm. Nút thắt: ta **thử lại cả mảng** sau mỗi lần sửa dù chỉ có **một cặp giảm** đáng quan tâm. Ta cần chỉ ra ngay **vị trí đáng sửa** và **cách sửa đúng**.

## 6. Ý tưởng thuật toán / bước đột phá

Duyệt từ trái sang phải. Ở cặp giảm đầu tiên `(a, b) = (nums[i], nums[i+1])`:

1. Phần `nums[0..i]` đã không giảm (chưa gặp cặp giảm trước đó).
2. Bắt buộc phải sửa `a` hoặc `b`.
3. **Hạ `a` xuống `b`** là lựa chọn **tốt nhất nếu hợp lệ**: nó giữ nguyên `b` và biến `a` thành giá trị nhỏ nhất có thể (`b`), nên không làm khó phần tương lai. Điều kiện hợp lệ: `i == 0` hoặc `nums[i-1] <= b`.
4. Nếu `nums[i-1] > b` thì hạ `a` không giúp được (vì `a'` phải `>= nums[i-1] > b`). Ta **buộc phải nâng `b` lên `a`** (`b' = a` là giá trị nhỏ nhất đủ dùng).
5. Sau khi sửa, tiếp tục duyệt; nếu gặp thêm **cặp giảm thứ hai** thì trả `false`.

Mỗi phần tử xử lý `O(1)` → `O(n)`.

## 7. Các cách tiếp theo

Tổng cộng 5 cách: (1) thử sửa từng vị trí (mục 5), (2) greedy đếm cặp giảm và sửa tại chỗ, (3) sửa ở cặp giảm đầu tiên rồi kiểm tra cả mảng, (4) so sánh lân cận không đụng mảng, (5) dãy con không giảm dài nhất.

### 7.1. Cách 2 — Greedy: đếm cặp giảm, sửa tại chỗ `O(n)`

**Giải thích từng đoạn.**

1. `cnt` đếm số lần phải sửa. Duyệt `i` từ `0` đến `n - 2`.
2. Khi `nums[i] > nums[i+1]`: tăng `cnt`; nếu `cnt > 1` trả `False`.
3. Nếu `i > 0` **và** `nums[i-1] > nums[i+1]`: không thể hạ `nums[i]`, nên **nâng** `nums[i+1] = nums[i]`.
4. Ngược lại (`i == 0` hoặc `nums[i-1] <= nums[i+1]`): **hạ** `nums[i] = nums[i+1]`.
5. Hết vòng lặp trả `True`.

Việc sửa thật vào mảng đảm bảo các so sánh ở các bước sau dùng đúng "mảng sau khi sửa".

In [9]:
def check_greedy(nums):
    nums = nums[:]                       # không đổi mảng của người gọi
    cnt = 0
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            cnt += 1
            if cnt > 1:
                return False
            if i > 0 and nums[i - 1] > nums[i + 1]:
                nums[i + 1] = nums[i]    # buộc phải nâng bên phải
            else:
                nums[i] = nums[i + 1]    # hạ bên trái
    return True


for nums, expected in examples:
    print(nums, "->", check_greedy(nums))
    assert check_greedy(nums) == expected

[4, 2, 3] -> True
[4, 2, 1] -> False


**Code nộp LeetCode (cách 2):**

```python
class Solution:
    def checkPossibility(self, nums: List[int]) -> bool:
        cnt = 0
        for i in range(len(nums) - 1):
            if nums[i] > nums[i + 1]:
                cnt += 1
                if cnt > 1:
                    return False
                if i > 0 and nums[i - 1] > nums[i + 1]:
                    nums[i + 1] = nums[i]
                else:
                    nums[i] = nums[i + 1]
        return True
```

**Độ phức tạp:** Time `O(n)`, Space `O(1)` (sửa tại chỗ; bản trong notebook sao chép mảng nên `O(n)`).

### 7.2. Cách 3 — Sửa ở cặp giảm ĐẦU TIÊN, thử cả hai hướng rồi kiểm tra toàn mảng

**Giải thích từng đoạn.**

1. Tìm cặp giảm đầu tiên `i` (`nums[i] > nums[i+1]`). Nếu không có thì trả `True`.
2. Thử **kịch bản A**: `nums[i] = nums[i+1]` (hạ trái); kiểm tra cả mảng không giảm.
3. Nếu A không được, thử **kịch bản B**: `nums[i+1] = nums[i]` (nâng phải); kiểm tra cả mảng.
4. Trả kết quả của B.

Cách này **không cần suy nghĩ về điều kiện `nums[i-1] <= nums[i+1]`**; ta giao việc cho hàm kiểm tra. Gọn và khó sai, đổi lại tốn thêm một lần quét `O(n)` (vẫn `O(n)` tổng).

In [10]:
def check_first_descent(nums):
    n = len(nums)
    for i in range(n - 1):
        if nums[i] > nums[i + 1]:
            a = nums[:]
            a[i] = nums[i + 1]               # kịch bản A: hạ trái
            if is_nondecreasing(a):
                return True
            b = nums[:]
            b[i + 1] = nums[i]               # kịch bản B: nâng phải
            return is_nondecreasing(b)
    return True


for nums, expected in examples:
    print(nums, "->", check_first_descent(nums))
    assert check_first_descent(nums) == expected

[4, 2, 3] -> True
[4, 2, 1] -> False


**Code nộp LeetCode (cách 3):**

```python
class Solution:
    def checkPossibility(self, nums: List[int]) -> bool:
        def is_sorted(a):
            return all(a[i] <= a[i + 1] for i in range(len(a) - 1))

        for i in range(len(nums) - 1):
            if nums[i] > nums[i + 1]:
                a = nums[:]
                a[i] = nums[i + 1]
                if is_sorted(a):
                    return True
                b = nums[:]
                b[i + 1] = nums[i]
                return is_sorted(b)
        return True
```

**Độ phức tạp:** Time `O(n)` (một lần tìm cặp giảm, tối đa hai lần sao chép và kiểm tra), Space `O(n)` cho hai bản sao.

### 7.3. Cách 4 — So sánh lân cận, không sửa mảng `O(n)` / `O(1)`

**Ý tưởng.** Nếu chỉ có **một** cặp giảm tại `i`, thì phần `nums[0..i]` không giảm và phần `nums[i+1..]` cũng không giảm. Khi đó "có sửa được không" chỉ phụ thuộc hai điều kiện cục bộ:

- `trai_ok = (i == 0) or nums[i-1] <= nums[i+1]` (hạ `nums[i]` được);
- `phai_ok = (i + 2 >= n) or nums[i] <= nums[i+2]` (nâng `nums[i+1]` được).

Mảng sửa được nếu **tối đa một cặp giảm** và tại cặp đó `trai_ok or phai_ok`. Không cần động vào mảng, nên cũng không lo "mảng bị đổi" cho các lời gọi sau.

In [11]:
def check_local(nums):
    n = len(nums)
    cnt = 0
    for i in range(n - 1):
        if nums[i] > nums[i + 1]:
            cnt += 1
            if cnt > 1:
                return False
            trai_ok = (i == 0) or nums[i - 1] <= nums[i + 1]
            phai_ok = (i + 2 >= n) or nums[i] <= nums[i + 2]
            if not (trai_ok or phai_ok):
                return False
    return True


for nums, expected in examples:
    print(nums, "->", check_local(nums))
    assert check_local(nums) == expected

[4, 2, 3] -> True
[4, 2, 1] -> False


**Code nộp LeetCode (cách 4):**

```python
class Solution:
    def checkPossibility(self, nums: List[int]) -> bool:
        n = len(nums)
        cnt = 0
        for i in range(n - 1):
            if nums[i] > nums[i + 1]:
                cnt += 1
                if cnt > 1:
                    return False
                trai_ok = i == 0 or nums[i - 1] <= nums[i + 1]
                phai_ok = i + 2 >= n or nums[i] <= nums[i + 2]
                if not (trai_ok or phai_ok):
                    return False
        return True
```

**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### 7.4. Cách 5 — Dãy con không giảm dài nhất (LNDS), `O(n log n)`

**Ý tưởng khác hẳn.** Ta **được đổi một phần tử thành số nguyên tùy ý**. Gọi `L` là độ dài **dãy con không giảm dài nhất** (LNDS — các phần tử không nhất thiết liền nhau). Khi đó:

> Sửa được bằng tối đa 1 lần  ⇔  `n - L <= 1`.

Lý do: (⇒) các phần tử **không bị sửa** tạo thành một dãy con không giảm, nên `L >= n - 1`. (⇐) nếu `L >= n - 1`, giữ lại dãy con đó; với phần tử còn lại (nếu có) ta đặt nó bằng **giá trị của hàng xóm trái** trong dãy giữ lại (hoặc hàng xóm phải nếu nó ở đầu), nằm giữa hai phần tử giữ lại kề nó, nên mảng vẫn không giảm.

Tính LNDS bằng thuật toán "tails + bisect_right" (mỗi phần tử một lần tìm nhị phân): `tails[k]` là phần tử cuối nhỏ nhất của dãy con không giảm có độ dài `k + 1`. Dùng `bisect_right` (chứ không phải `bisect_left`) để cho phép **bằng nhau** nối tiếp nhau.

In [12]:
def lnds_length(nums):
    tails = []
    for x in nums:
        p = bisect_right(tails, x)
        if p == len(tails):
            tails.append(x)
        else:
            tails[p] = x
    return len(tails)


def check_lnds(nums):
    return len(nums) - lnds_length(nums) <= 1


print("LNDS của [4,2,3] =", lnds_length([4, 2, 3]), "| của [4,2,1] =", lnds_length([4, 2, 1]), "| của [1,2,2,3] =", lnds_length([1, 2, 2, 3]))
for nums, expected in examples:
    print(nums, "->", check_lnds(nums))
    assert check_lnds(nums) == expected

LNDS của [4,2,3] = 2 | của [4,2,1] = 1 | của [1,2,2,3] = 4
[4, 2, 3] -> True
[4, 2, 1] -> False


**Code nộp LeetCode (cách 5):**

```python
from bisect import bisect_right

class Solution:
    def checkPossibility(self, nums: List[int]) -> bool:
        tails = []
        for x in nums:
            p = bisect_right(tails, x)
            if p == len(tails):
                tails.append(x)
            else:
                tails[p] = x
        return len(nums) - len(tails) <= 1
```

**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`. Cách này chậm hơn greedy nhưng **tổng quát hóa ngay** cho "tối đa `k` lần sửa" (mục 15).

Năm cách cho cùng đáp án; mục 12 sẽ đối chiếu tất cả bằng `assert` với oracle vét cạn.

## 8. TRACE từng bước

Trace greedy (cách 2). Hàm `trace_greedy` in từng cặp, đánh dấu cặp giảm, và nói rõ ta **hạ trái** hay **nâng phải** cùng mảng sau khi sửa.

Bảng tay cho `nums = [3, 4, 2, 5]` (đáp án `true`):

| `i` | `(nums[i], nums[i+1])` | cặp giảm? | `cnt` | `nums[i-1] > nums[i+1]`? | hành động | mảng |
|---|---|---|---|---|---|---|
| 0 | `(3, 4)` | không | 0 | - | - | `[3,4,2,5]` |
| 1 | `(4, 2)` | **có** | 1 | `3 > 2` có | nâng `nums[2] = 4` | `[3,4,4,5]` |
| 2 | `(4, 5)` | không | 1 | - | - | `[3,4,4,5]` |

Kết quả `true`.

In [13]:
def trace_greedy(nums):
    nums = nums[:]
    cnt = 0
    print("bắt đầu:", nums)
    for i in range(len(nums) - 1):
        a, b = nums[i], nums[i + 1]
        if a <= b:
            print(f"i={i}: ({a},{b}) ok            cnt={cnt} mảng={nums}")
            continue
        cnt += 1
        if cnt > 1:
            print(f"i={i}: ({a},{b}) GIẢM lần {cnt} -> False")
            return False
        if i > 0 and nums[i - 1] > b:
            nums[i + 1] = a
            action = f"nums[{i-1}]={nums[i-1]} > {b} nên NÂNG nums[{i+1}] = {a}"
        else:
            nums[i] = b
            action = f"HẠ nums[{i}] = {b}"
        print(f"i={i}: ({a},{b}) GIẢM lần {cnt} -> {action} | mảng={nums}")
    print("không còn cặp giảm thứ hai -> True")
    return True


assert trace_greedy([3, 4, 2, 5]) is True
print()
assert trace_greedy([4, 2, 3]) is True
print()
assert trace_greedy([3, 4, 2, 3]) is False

bắt đầu: [3, 4, 2, 5]
i=0: (3,4) ok            cnt=0 mảng=[3, 4, 2, 5]
i=1: (4,2) GIẢM lần 1 -> nums[0]=3 > 2 nên NÂNG nums[2] = 4 | mảng=[3, 4, 4, 5]
i=2: (4,5) ok            cnt=1 mảng=[3, 4, 4, 5]
không còn cặp giảm thứ hai -> True

bắt đầu: [4, 2, 3]
i=0: (4,2) GIẢM lần 1 -> HẠ nums[0] = 2 | mảng=[2, 2, 3]
i=1: (2,3) ok            cnt=1 mảng=[2, 2, 3]
không còn cặp giảm thứ hai -> True

bắt đầu: [3, 4, 2, 3]
i=0: (3,4) ok            cnt=0 mảng=[3, 4, 2, 3]
i=1: (4,2) GIẢM lần 1 -> nums[0]=3 > 2 nên NÂNG nums[2] = 4 | mảng=[3, 4, 4, 3]
i=2: (4,3) GIẢM lần 2 -> False


Thêm một ca có số âm và số bằng nhau: `[-1, 4, 2, 3]` (hạ `4` xuống `2`) và `[2, 3, 3, 2, 4]` (cặp giảm `(3, 2)` tại `i = 2`; `nums[1] = 3 > 2` nên nâng `2` lên `3`).

In [14]:
assert trace_greedy([-1, 4, 2, 3]) is True
print()
assert trace_greedy([2, 3, 3, 2, 4]) is True
print()
assert trace_greedy([1, 2, 4, 5, 3, 6]) is True

bắt đầu: [-1, 4, 2, 3]
i=0: (-1,4) ok            cnt=0 mảng=[-1, 4, 2, 3]
i=1: (4,2) GIẢM lần 1 -> HẠ nums[1] = 2 | mảng=[-1, 2, 2, 3]
i=2: (2,3) ok            cnt=1 mảng=[-1, 2, 2, 3]
không còn cặp giảm thứ hai -> True

bắt đầu: [2, 3, 3, 2, 4]
i=0: (2,3) ok            cnt=0 mảng=[2, 3, 3, 2, 4]
i=1: (3,3) ok            cnt=0 mảng=[2, 3, 3, 2, 4]
i=2: (3,2) GIẢM lần 1 -> nums[1]=3 > 2 nên NÂNG nums[3] = 3 | mảng=[2, 3, 3, 3, 4]
i=3: (3,4) ok            cnt=1 mảng=[2, 3, 3, 3, 4]
không còn cặp giảm thứ hai -> True

bắt đầu: [1, 2, 4, 5, 3, 6]
i=0: (1,2) ok            cnt=0 mảng=[1, 2, 4, 5, 3, 6]
i=1: (2,4) ok            cnt=0 mảng=[1, 2, 4, 5, 3, 6]
i=2: (4,5) ok            cnt=0 mảng=[1, 2, 4, 5, 3, 6]
i=3: (5,3) GIẢM lần 1 -> nums[2]=4 > 3 nên NÂNG nums[4] = 5 | mảng=[1, 2, 4, 5, 5, 6]
i=4: (5,6) ok            cnt=1 mảng=[1, 2, 4, 5, 5, 6]
không còn cặp giảm thứ hai -> True


## 9. Code nộp LeetCode

Đây là cách greedy tại chỗ (cách 2) — ngắn nhất, `O(n)` thời gian, `O(1)` bộ nhớ.

```python
from typing import List

class Solution:
    def checkPossibility(self, nums: List[int]) -> bool:
        cnt = 0
        for i in range(len(nums) - 1):
            if nums[i] > nums[i + 1]:
                cnt += 1
                if cnt > 1:
                    return False
                if i > 0 and nums[i - 1] > nums[i + 1]:
                    nums[i + 1] = nums[i]     # buộc nâng bên phải
                else:
                    nums[i] = nums[i + 1]     # hạ bên trái
        return True
```

In [15]:
class Solution:
    def checkPossibility(self, nums: List[int]) -> bool:
        cnt = 0
        for i in range(len(nums) - 1):
            if nums[i] > nums[i + 1]:
                cnt += 1
                if cnt > 1:
                    return False
                if i > 0 and nums[i - 1] > nums[i + 1]:
                    nums[i + 1] = nums[i]
                else:
                    nums[i] = nums[i + 1]
        return True


sol = Solution()
for nums, expected in examples:
    got = sol.checkPossibility(nums[:])
    print(nums, "->", got, "| kỳ vọng", expected)
    assert got == expected

[4, 2, 3] -> True | kỳ vọng True
[4, 2, 1] -> False | kỳ vọng False


## 10. Vì sao đúng

**Phát biểu.** Greedy trả `True` ⇔ mảng có thể thành không giảm bằng tối đa một lần sửa.

**Chứng minh ngắn.**

1. Nếu không có cặp giảm: đã không giảm → `True` (đúng).
2. Nếu có **hai cặp giảm trở lên** sau khi đã sửa tại cặp đầu theo cách tốt nhất (xem 4–5), ta trả `False`. Cần chứng minh lựa chọn ở bước 3 là **tốt nhất** nên "còn cặp giảm sau khi sửa" nghĩa là thật sự không sửa được.
3. Xét cặp giảm đầu tiên `(a, b) = (nums[i], nums[i+1])`. Mọi cách sửa hợp lệ phải đổi `a` hoặc `b` (đổi phần tử khác không làm đổi quan hệ `a > b`).
4. **Trường hợp `i == 0` hoặc `nums[i-1] <= b`:** hạ `a` thành `b` hợp lệ về cặp `(i-1, i)` và `(i, i+1)`. Nó **tốt hơn hoặc bằng** mọi cách sửa khác, vì `b` giữ nguyên (không làm khó cặp `(i+1, i+2)`), còn mọi cách sửa bằng nâng `b` làm `b` lớn hơn, chỉ gây thêm rủi ro với `nums[i+2]`. Do đó nếu có lời giải thì lời giải "hạ `a`" cũng là lời giải.
5. **Trường hợp `nums[i-1] > b`:** nếu hạ `a` thành `a'` thì cần `a' >= nums[i-1] > b` (để khớp với bên trái) và đồng thời `a' <= b` (để khớp với `b`) → mâu thuẫn. Vậy buộc phải nâng `b` lên `b' >= a`; chọn `b' = a` (nhỏ nhất) là tốt nhất cho phần sau.
6. Sau khi sửa, theo giả thiết quy nạp, phần còn lại không cần sửa thêm; nếu còn cặp giảm nữa (đếm `cnt > 1`) thì chắc chắn cần ≥ 2 lần sửa (vì một lần sửa tốt nhất cũng không đủ).

**Với cách 4:** sau khi biết chỉ có một cặp giảm, hai vế của nó đều không giảm; cặp này được sửa nếu hạ trái hoặc nâng phải hợp lệ, đúng bằng hai điều kiện `trai_ok`, `phai_ok`.

**Với cách 5:** tương đương `n - LNDS <= 1` như đã chứng minh ở mục 7.4.

## 11. Time/Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| `oracle_nd` | `O(n^2 * V)` | `O(n)` | oracle chậm |
| 1. `check_try_each` | `O(n^2)` | `O(n)` | TLE với `n = 10^4` |
| 2. greedy tại chỗ (`Solution`) | `O(n)` | `O(1)` | cách nộp |
| 3. sửa cặp giảm đầu tiên | `O(n)` | `O(n)` | dễ viết đúng nhất |
| 4. so sánh lân cận | `O(n)` | `O(1)` | không đổi mảng |
| 5. LNDS | `O(n log n)` | `O(n)` | tổng quát cho `k` lần sửa |

Greedy đúng một lượt quét, mỗi phần tử so sánh `O(1)` lần: `O(n)`. Với `n = 10^4`, chạy trong tích tắc.

## 12. Bộ test

Chiến lược: (a) ví dụ của đề, (b) edge case (1–2 phần tử, đã không giảm, toàn bằng nhau, cặp giảm ở đầu/cuối), (c) phản ví dụ cho các lời giải sai, (d) đối chiếu **mọi cách** với `oracle_nd` trên rất nhiều mảng ngẫu nhiên có seed, (e) test lớn `n = 10^4`.

Lưu ý: cách 2 và `Solution` **sửa tại chỗ**, nên khi gọi ta luôn truyền một bản sao `a[:]` để không làm hỏng dữ liệu test.

### 12.1. Ví dụ của đề và edge case

In [16]:
ALL = {
    "try_each": check_try_each,
    "greedy": check_greedy,
    "first_descent": check_first_descent,
    "local": check_local,
    "lnds": check_lnds,
    "solution": lambda a: Solution().checkPossibility(a[:]),
}

cases = [
    ([4, 2, 3], True),                  # ví dụ 1
    ([4, 2, 1], False),                 # ví dụ 2
    ([1], True),                        # một phần tử
    ([2, 1], True),                     # hai phần tử luôn được
    ([1, 2, 3, 4], True),               # đã không giảm
    ([5, 5, 5, 5], True),               # toàn bằng nhau
    ([3, 4, 2, 3], False),              # một cặp giảm nhưng vẫn false
    ([3, 4, 2, 5], True),               # phải nâng bên phải
    ([5, 7, 1, 8], True),
    ([1, 2, 4, 5, 3, 6], True),
    ([-1, 4, 2, 3], True),              # có số âm
    ([2, 3, 3, 2, 4], True),            # có số bằng nhau
    ([1, 3, 2, 1], False),
    ([4, 2, 3, 1], False),
    ([10, 1, 2, 3], True),              # cặp giảm ở đầu
    ([1, 2, 3, 10, 4], True),           # cặp giảm ở cuối
    ([1, 2, 5, 3, 3], True),            # nâng phải bằng nhau
    ([100000, -100000], True),          # biên giá trị
    ([1, 1, 1, 0, 1, 1, 1], True),      # hạ/nâng với số bằng nhau
]
for nums, expected in cases:
    assert oracle_nd(nums) == expected, (nums, oracle_nd(nums), expected)
    for name, fn in ALL.items():
        got = fn(nums)
        assert got == expected, (name, nums, got, expected)
print("tất cả", len(cases), "ca cố định đều đúng với", len(ALL), "cách")

tất cả 19 ca cố định đều đúng với 6 cách


### 12.2. Phản ví dụ cho các lời giải sai

Các hàm sai được trình bày chi tiết ở mục 13; ở đây ta ghi lại ca làm lộ từng lỗi để dùng làm test hồi quy:

- `[3, 4, 2, 3]` → đúng là `False` (chỉ đếm cặp giảm sẽ ra `True`).
- `[3, 4, 2, 5]` → đúng là `True` (luôn hạ trái sẽ ra `False`).
- `[4, 2, 3]` → đúng là `True` (luôn nâng phải sẽ ra `False`).

In [17]:
regression = [([3, 4, 2, 3], False), ([3, 4, 2, 5], True), ([4, 2, 3], True), ([1, 1, 1], True)]
for nums, expected in regression:
    assert oracle_nd(nums) == expected
    for name, fn in ALL.items():
        assert fn(nums) == expected, (name, nums)
print("các ca hồi quy đều đúng:", [n for n, _ in regression])

các ca hồi quy đều đúng: [[3, 4, 2, 3], [3, 4, 2, 5], [4, 2, 3], [1, 1, 1]]


### 12.3. Đối chiếu ngẫu nhiên nhỏ (giá trị trong `[-3, 3]` để có nhiều số bằng nhau)

In [18]:
rng = random.Random(665)
checked = 0
true_count = 0
for _ in range(3000):
    n = rng.randint(1, 8)
    nums = [rng.randint(-3, 3) for _ in range(n)]
    expected = oracle_nd(nums)
    true_count += expected
    for name, fn in ALL.items():
        assert fn(nums) == expected, (name, nums, fn(nums), expected)
    checked += 1
print("đã đối chiếu", checked, "mảng ngẫu nhiên nhỏ | số ca True:", true_count, "| số ca False:", checked - true_count)
assert 0 < true_count < checked          # cả hai nhãn đều xuất hiện

đã đối chiếu

 3000 mảng ngẫu nhiên nhỏ | số ca True: 1426 | số ca False: 1574


### 12.4. Đối chiếu có cấu trúc: mảng "gần không giảm" (đúng ràng buộc giá trị)

Mảng ngẫu nhiên thuần thường có rất nhiều cặp giảm nên đa số trả `False`. Ta sinh mảng **đã sắp xếp** rồi **phá hỏng** đúng 0, 1 hoặc 2 phần tử để có nhiều ca `True` khó (đáp án sát ranh giới).

In [19]:
def near_sorted(rng, n, breaks):
    a = sorted(rng.randint(-10 ** 5, 10 ** 5) for _ in range(n))
    for _ in range(breaks):
        i = rng.randrange(n)
        a[i] = rng.randint(-10 ** 5, 10 ** 5)
    return a


rng = random.Random(1234)
stats = {True: 0, False: 0}
for _ in range(400):
    n = rng.randint(1, 120)
    nums = near_sorted(rng, n, rng.choice([0, 1, 1, 2]))
    expected = check_try_each(nums)           # O(n^2), nhanh với n <= 120
    stats[expected] += 1
    for name, fn in ALL.items():
        assert fn(nums) == expected, (name, nums)
print("400 mảng gần-sắp-xếp khớp | True:", stats[True], "False:", stats[False])
assert stats[True] > 0 and stats[False] > 0

# phá hỏng bằng hoán vị hai phần tử kề nhau và nhiều giá trị trùng
rng = random.Random(55)
for _ in range(400):
    n = rng.randint(2, 60)
    a = sorted(rng.randint(0, 6) for _ in range(n))
    i = rng.randrange(n - 1)
    a[i], a[i + 1] = a[i + 1], a[i]
    expected = oracle_nd(a) if n <= 12 else check_try_each(a)
    for name, fn in ALL.items():
        assert fn(a) == expected, (name, a)
print("400 mảng hoán vị cặp kề khớp")

400 mảng gần-sắp-xếp khớp | True: 308 False: 92
400 mảng hoán vị cặp kề khớp


### 12.5. Test kích thước lớn (`n = 10^4`)

Chỉ chạy các cách `O(n)` và `O(n log n)`. Ta đối chiếu chúng với nhau và với đáp án mà ta biết trước theo cách dựng.

In [20]:
import time

n = 10 ** 4
rng = random.Random(99)
base = sorted(rng.randint(-10 ** 5, 10 ** 5) for _ in range(n))
fast = {k: v for k, v in ALL.items() if k != "try_each"}
t = time.perf_counter()

# 1) đã không giảm -> True
assert all(fn(base) for fn in fast.values())

# 2) hạ một phần tử ở giữa xuống rất thấp -> True nếu hai bên vẫn khớp (cần nâng/hạ đúng cách)
low = base[:]
low[n // 2] = -10 ** 5 - 1000
assert all(fn(low) for fn in fast.values())

# 3) hai ổ gà cách xa nhau -> False
two = base[:]
two[100] = 10 ** 5
two[5000] = -10 ** 5
res = {k: fn(two) for k, fn in fast.items()}
assert set(res.values()) == {False}, res

# 4) toàn giảm dần -> False
desc = list(range(n, 0, -1))
assert set(fn(desc) for fn in fast.values()) == {False}

# 5) toàn bằng nhau -> True
assert all(fn([7] * n) for fn in fast.values())
print("test lớn n = 10^4 đều đúng, thời gian:", round(time.perf_counter() - t, 2), "s")

test lớn n = 10^4 đều đúng, thời gian: 0.02 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đếm số cặp giảm (`<= 1` là xong)

In [21]:
def wrong_count_only(nums):
    cnt = 0
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            cnt += 1
    return cnt <= 1


print("sai :", wrong_count_only([3, 4, 2, 3]), "| đúng:", check_greedy([3, 4, 2, 3]))
assert wrong_count_only([3, 4, 2, 3]) is True and check_greedy([3, 4, 2, 3]) is False

sai : True | đúng: False


**Vì sao sai:** `[3, 4, 2, 3]` có đúng một cặp giảm `(4, 2)`, nhưng hạ `4` xuống `2` làm hỏng cặp `(3, 2)`, còn nâng `2` lên `4` làm hỏng cặp `(4, 3)`. Phải xét **cả hai hàng xóm** của cặp giảm. **Cách đúng:** thêm điều kiện lân cận (`trai_ok or phai_ok`).

### Lỗi 2 — Luôn hạ bên trái (`nums[i] = nums[i+1]`)

In [22]:
def wrong_always_lower(nums):
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            a = nums[:]
            a[i] = nums[i + 1]               # SAI: chỉ thử hạ trái
            return is_nondecreasing(a)
    return True


print("sai :", wrong_always_lower([3, 4, 2, 5]), "| đúng:", check_greedy([3, 4, 2, 5]))
assert wrong_always_lower([3, 4, 2, 5]) is False and check_greedy([3, 4, 2, 5]) is True

sai : False | đúng: True


**Vì sao sai:** ở `[3, 4, 2, 5]`, hạ `4` xuống `2` cho `[3, 2, 2, 5]` — lại tạo cặp giảm mới `(3, 2)`. Chỉ có **nâng `2` lên `4`** mới đúng. **Cách đúng:** nếu `nums[i-1] > nums[i+1]` thì nâng phải.

### Lỗi 3 — Luôn nâng bên phải (`nums[i+1] = nums[i]`)

In [23]:
def wrong_always_raise(nums):
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            a = nums[:]
            a[i + 1] = nums[i]               # SAI: chỉ thử nâng phải
            return is_nondecreasing(a)
    return True


print("sai :", wrong_always_raise([4, 2, 3]), "| đúng:", check_greedy([4, 2, 3]))
assert wrong_always_raise([4, 2, 3]) is False and check_greedy([4, 2, 3]) is True

sai : False | đúng: True


**Vì sao sai:** nâng `2` lên `4` được `[4, 4, 3]` — lại giảm ở `(4, 3)`. Trong khi hạ `4` xuống `2` cho `[2, 2, 3]` hợp lệ. **Cách đúng:** thử cả hai hướng (cách 3) hoặc chọn theo điều kiện lân cận (cách 2, 4).

### Lỗi 4 — Quên chặn `i > 0` (bẫy `nums[-1]`)

In [24]:
def wrong_index_wrap(nums):
    nums = nums[:]
    cnt = 0
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            cnt += 1
            if cnt > 1:
                return False
            if nums[i - 1] > nums[i + 1]:    # SAI: thiếu 'i > 0 and'; i = 0 đọc nums[-1]
                nums[i + 1] = nums[i]
            else:
                nums[i] = nums[i + 1]
    return True


print("sai :", wrong_index_wrap([4, 2, 3]), "| đúng:", check_greedy([4, 2, 3]))
assert wrong_index_wrap([4, 2, 3]) is False and check_greedy([4, 2, 3]) is True

sai : False | đúng: True


**Vì sao sai:** ở `i = 0`, `nums[i - 1]` là `nums[-1] = 3` (phần tử cuối!), nên `3 > 2` "đúng" và ta nâng nhầm `2` lên `4`. Python **không báo lỗi**. **Cách đúng:** `if i > 0 and nums[i - 1] > nums[i + 1]`.

### Lỗi 5 — Dùng `>=` thay vì `>` (quên rằng bằng nhau vẫn được)

In [25]:
def wrong_ge(nums):
    cnt = 0
    for i in range(len(nums) - 1):
        if nums[i] >= nums[i + 1]:           # SAI: bằng nhau cũng bị tính là giảm
            cnt += 1
    return cnt <= 1


print("sai :", wrong_ge([1, 1, 1]), "| đúng:", check_greedy([1, 1, 1]))
assert wrong_ge([1, 1, 1]) is False and check_greedy([1, 1, 1]) is True

sai : False | đúng: True


**Vì sao sai:** đề định nghĩa "không giảm" là `nums[i] <= nums[i + 1]`, nên hai số bằng nhau **không** phải cặp giảm. **Cách đúng:** chỉ đếm khi `nums[i] > nums[i + 1]`.

### Lỗi 6 (bonus) — Sửa vào mảng đầu vào rồi dùng lại
Cách nộp `Solution` sửa trực tiếp `nums`. Trên LeetCode điều đó ổn, nhưng khi **tự test** nhiều hàm trên cùng một mảng thì hàm đầu tiên làm hỏng dữ liệu cho hàm sau. Hãy luôn truyền bản sao `nums[:]` (như ta làm ở `ALL`).

In [26]:
data = [3, 4, 2, 5]
Solution().checkPossibility(data)
print("sau khi Solution chạy, mảng gốc bị đổi thành:", data)
assert data != [3, 4, 2, 5]

sau khi Solution chạy, mảng gốc bị đổi thành: [3, 4, 4, 5]


## 14. Mẹo & nhận dạng pattern

**Dấu hiệu nhận ra bài "sửa tối đa một phần tử":**

- "Tối đa một thay đổi" → tập trung vào **cặp vi phạm đầu tiên**; hai vi phạm "độc lập" thì thất bại ngay.
- Bài sắp xếp/đơn điệu → nghĩ tới **cặp lân cận** và **hai hàng xóm** của chúng.

**Mẹo nhớ:**

1. Ở cặp giảm `(a, b)`: ưu tiên **hạ `a`** (vì giữ `b` nguyên, ít rủi ro hơn); chỉ **nâng `b`** khi `nums[i-1] > b` (buộc).
2. Luôn nghĩ ở `i = 0` và `i = n - 2`: không có hàng xóm trái/phải thì điều kiện tương ứng **mặc định đúng**.
3. Khi không chắc greedy, dùng **cách 3 (thử hai kịch bản rồi kiểm tra toàn mảng)** — gần như không thể sai.
4. Cần "tối đa `k` lần sửa" → nhớ `n - LNDS <= k`.
5. Tự test bằng oracle vét cạn trên mảng nhỏ giá trị bé (nhiều số bằng nhau) — nơi lỗi `>` vs `>=` lộ ra.
6. Python: `nums[i-1]` ở `i = 0` là phần tử **cuối** — chặn `i > 0` trước.

## 15. Biến thể

### 15.1. Số lần sửa **tối thiểu** để thành không giảm (và "tối đa `k` lần")

Từ mục 7.4: số lần sửa tối thiểu là `n - LNDS`. Ta kiểm chứng với một **DP độc lập**: `f[i][v]` = số lần sửa tối thiểu cho `i` phần tử đầu sao cho phần tử thứ `i` có giá trị `v` và dãy không giảm.

In [27]:
def min_changes(nums):
    return len(nums) - lnds_length(nums)


def min_changes_dp(nums):
    """Oracle: DP trên miền giá trị [min, max]."""
    lo, hi = min(nums), max(nums)
    values = range(lo, hi + 1)
    prev = {v: (0 if nums[0] == v else 1) for v in values}
    for x in nums[1:]:
        cur = {}
        best_so_far = None
        for v in values:                     # best_so_far = min prev[u] với u <= v
            best_so_far = prev[v] if best_so_far is None else min(best_so_far, prev[v])
            cur[v] = best_so_far + (0 if x == v else 1)
        prev = cur
    return min(prev.values())


def at_most_k_changes(nums, k):
    return min_changes(nums) <= k


assert min_changes([4, 2, 3]) == 1 and min_changes([4, 2, 1]) == 2 and min_changes([1, 2, 3]) == 0
rng = random.Random(151)
for _ in range(500):
    arr = [rng.randint(-4, 4) for _ in range(rng.randint(1, 9))]
    assert min_changes(arr) == min_changes_dp(arr), arr
    assert at_most_k_changes(arr, 1) == check_greedy(arr)
print("15.1 OK:", min_changes([3, 4, 2, 3]), min_changes([5, 4, 3, 2, 1]))

15.1 OK: 2 4


### 15.2. Phiên bản **không tăng** (non-increasing)

Đổi dấu mọi phần tử: mảng không tăng ⇔ mảng đối dấu không giảm.

In [28]:
def check_nonincreasing(nums):
    return check_greedy([-x for x in nums])


assert check_nonincreasing([1, 4, 3]) is True       # sửa 1 thành 4 được [4, 4, 3]
assert check_nonincreasing([1, 2, 3]) is False
assert check_nonincreasing([5, 5, 5]) is True
print(check_nonincreasing([1, 4, 3]), check_nonincreasing([1, 2, 3]))

True False


### 15.3. Phiên bản **xóa** tối đa một phần tử (thay vì sửa)

Với "không giảm", xóa một phần tử và sửa một phần tử cho **cùng** đáp án: phần tử bị sửa có thể gán bằng hàng xóm, tương đương bị "hòa vào" dãy. Kiểm chứng bằng vét cạn.

In [29]:
def check_delete_one(nums):
    if is_nondecreasing(nums):
        return True
    return any(is_nondecreasing(nums[:i] + nums[i + 1:]) for i in range(len(nums)))


rng = random.Random(152)
for _ in range(1000):
    arr = [rng.randint(-3, 3) for _ in range(rng.randint(1, 9))]
    assert check_delete_one(arr) == check_greedy(arr), arr
print("15.3 OK: xóa 1 phần tử ≡ sửa 1 phần tử (với không giảm)")

15.3 OK: xóa 1 phần tử ≡ sửa 1 phần tử (với không giảm)


### 15.4. Phiên bản **tăng chặt** (`nums[i] < nums[i+1]`) với số nguyên

Đặt `b[i] = nums[i] - i`. Dãy `nums` tăng chặt trên số nguyên ⇔ `b` không giảm (vì `nums[i+1] - nums[i] >= 1` ⇔ `b[i+1] >= b[i]`). Sửa một phần tử `nums[i]` thành số nguyên tùy ý tương ứng sửa `b[i]` thành số nguyên tùy ý. Vậy bài toán quy về bài gốc trên `b`.

In [30]:
def strict_one_change(nums):
    b = [x - i for i, x in enumerate(nums)]
    return check_greedy(b)


def strict_oracle(nums):
    def strictly(a):
        return all(a[i] < a[i + 1] for i in range(len(a) - 1))
    if strictly(nums):
        return True
    lo, hi, n = min(nums), max(nums), len(nums)
    for i in range(n):
        for v in range(lo - n - 2, hi + n + 3):
            a = nums[:]
            a[i] = v
            if strictly(a):
                return True
    return False


assert strict_one_change([1, 2, 2, 3]) is False
assert strict_one_change([1, 3, 3, 4]) is True
assert strict_one_change([5, 4, 3]) is False
rng = random.Random(154)
for _ in range(500):
    arr = [rng.randint(-3, 5) for _ in range(rng.randint(1, 7))]
    assert strict_one_change(arr) == strict_oracle(arr), arr
print("15.4 OK")

15.4 OK

Họ hàng của bài này: LeetCode 300 (Longest Increasing Subsequence), 673 (đếm số LIS), 1187 (Make Array Strictly Increasing — thay phần tử từ một tập cho trước, DP + bisect), 2111 (Minimum Operations to Make the Array K-Increasing — chia theo chỉ số rồi LNDS).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử sửa từng vị trí | thử đặt `nums[i]` bằng hai hàng xóm | `O(n^2)` | `O(n)` | oracle, `n` nhỏ |
| 2. Greedy tại chỗ | đếm cặp giảm; hạ trái hoặc nâng phải | `O(n)` | `O(1)` | **cách nộp** |
| 3. Sửa cặp giảm đầu tiên | thử hai kịch bản, kiểm tra cả mảng | `O(n)` | `O(n)` | khó sai nhất khi viết lúc thi |
| 4. So sánh lân cận | `trai_ok or phai_ok`, không đụng mảng | `O(n)` | `O(1)` | khi không được phép sửa input |
| 5. LNDS | `n - LNDS <= 1` | `O(n log n)` | `O(n)` | mở rộng cho `k` lần sửa |

Khuyến nghị: nộp cách 2 hoặc 4; dùng cách 3 để tự đối chiếu; dùng `oracle_nd` khi tự viết biến thể.

In [31]:
rng = random.Random(16)
for _ in range(500):
    nums = [rng.randint(-5, 5) for _ in range(rng.randint(1, 10))]
    res = {name: fn(nums) for name, fn in ALL.items()}
    res["oracle"] = oracle_nd(nums)
    assert len(set(res.values())) == 1, (nums, res)
print("500 mảng: 7 hàm cho cùng đáp án")

500 mảng: 7 hàm cho cùng đáp án


## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng sau, rồi chạy để kiểm tra:

```python
print(descents([3, 4, 2, 3]))
print(check_greedy([4, 2, 3]))
print(check_greedy([3, 4, 2, 3]))
```

In [32]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay greedy (dễ-vừa)

Với `nums = [5, 7, 1, 8]`, hãy cho biết: (a) chỉ số `i` của cặp giảm; (b) ta **hạ** hay **nâng**; (c) mảng sau khi sửa; (d) đáp án cuối cùng.

In [33]:
# Bài 2: điền kết quả tính tay
hand_index = None        # chỉ số i của cặp giảm
hand_action = None       # "hạ" hoặc "nâng"
hand_array = None        # mảng sau khi sửa, ví dụ [1, 2, 3]
hand_answer = None       # True/False

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành greedy tại chỗ.

In [34]:
def exercise_3_check(nums: List[int]) -> bool:
    cnt = 0
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            # TODO 1: tăng cnt; nếu cnt > 1 thì trả False
            # TODO 2: nếu i > 0 và nums[i - 1] > nums[i + 1] thì nâng nums[i + 1] = nums[i]
            # TODO 3: ngược lại hạ nums[i] = nums[i + 1]
            pass
    return None


# assert exercise_3_check([4, 2, 3]) is True
# assert exercise_3_check([4, 2, 1]) is False
# assert exercise_3_check([3, 4, 2, 3]) is False

### Bài 4 — Tự viết lại (vừa-khó): số lần sửa tối thiểu

Không nhìn lại mục 15.1, hãy tự viết `exercise_4_min_changes(nums)` trả về **số phần tử tối thiểu** cần sửa để mảng không giảm (sửa thành số nguyên tùy ý).

```python
exercise_4_min_changes([4, 2, 3]) -> 1
exercise_4_min_changes([4, 2, 1]) -> 2
exercise_4_min_changes([1, 2, 3]) -> 0
exercise_4_min_changes([3, 4, 2, 3]) -> 2
```

In [35]:
def exercise_4_min_changes(nums: List[int]) -> int:
    # TODO
    pass


# assert exercise_4_min_changes([4, 2, 3]) == 1
# assert exercise_4_min_changes([4, 2, 1]) == 2
# assert exercise_4_min_changes([1, 2, 3]) == 0
# assert exercise_4_min_changes([3, 4, 2, 3]) == 2

### Bài 5 — Mini challenge: tăng chặt với tối đa một lần sửa (khó)

Viết `exercise_5_strict(nums)` trả `True` nếu có thể biến mảng thành **tăng chặt** (`nums[i] < nums[i+1]`) bằng cách sửa tối đa một phần tử thành số nguyên tùy ý.

```python
exercise_5_strict([1, 2, 2, 3]) -> False
exercise_5_strict([1, 3, 3, 4]) -> True
exercise_5_strict([5, 4, 3])    -> False
exercise_5_strict([2, 2])       -> True
```

Gợi ý hướng nghĩ: biến đổi `b[i] = nums[i] - i`.

In [36]:
def exercise_5_strict(nums: List[int]) -> bool:
    # TODO
    pass


# assert exercise_5_strict([1, 2, 2, 3]) is False
# assert exercise_5_strict([1, 3, 3, 4]) is True
# assert exercise_5_strict([5, 4, 3]) is False
# assert exercise_5_strict([2, 2]) is True

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `descents` liệt kê chỉ số `i` mà `nums[i] > nums[i+1]`.
- Mức 2: `[3, 4, 2, 3]` chỉ giảm ở cặp `(4, 2)`, tức `i = 1`. Với `[3, 4, 2, 3]`, thử cả hạ lẫn nâng đều tạo cặp giảm mới.
- Mức 3: dòng 1 là `[1]`; dòng 2 là `True`; dòng 3 là `False`.

**Bài 2**
- Mức 1: tìm cặp giảm: `5 <= 7` ok, `7 > 1` giảm.
- Mức 2: kiểm tra `nums[i-1]` với `nums[i+1]`: `5 > 1`, nên không hạ được.
- Mức 3: (a) `1`; (b) nâng; (c) `[5, 7, 7, 8]`; (d) `True`.

**Bài 3**
- Mức 1: ba việc: đếm, rẽ nhánh nâng/hạ, trả kết quả.
- Mức 2: `cnt += 1`; `if cnt > 1: return False`; rẽ nhánh `if i > 0 and nums[i-1] > nums[i+1]`.
- Mức 3: nhớ `return True` sau vòng lặp (không phải `None`); nhớ biến `cnt` khởi tạo `0` bên ngoài vòng lặp.

**Bài 4**
- Mức 1: giữ lại càng nhiều phần tử càng tốt: dãy con không giảm dài nhất.
- Mức 2: tails + `bisect_right` cho LNDS.
- Mức 3: kết quả `len(nums) - len(tails)`.

**Bài 5**
- Mức 1: `nums[i] < nums[i+1]` trên số nguyên ⇔ `nums[i+1] - (i+1) >= nums[i] - i`.
- Mức 2: lập `b = [x - i for i, x in enumerate(nums)]`.
- Mức 3: trả `exercise_3_check` (hoặc greedy bất kỳ) áp dụng cho `b`; nhớ không sửa mảng gốc nếu bạn cần dùng lại.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [37]:
print(descents([3, 4, 2, 3]))
print(check_greedy([4, 2, 3]))
print(check_greedy([3, 4, 2, 3]))

assert descents([3, 4, 2, 3]) == [1]
assert check_greedy([4, 2, 3]) is True
assert check_greedy([3, 4, 2, 3]) is False

[1]
True
False


### Đáp án Bài 2

| `i` | `(nums[i], nums[i+1])` | giảm? | `nums[i-1] > nums[i+1]`? | hành động | mảng |
|---|---|---|---|---|---|
| 0 | `(5, 7)` | không | - | - | `[5,7,1,8]` |
| 1 | `(7, 1)` | có | `5 > 1` có | nâng `nums[2] = 7` | `[5,7,7,8]` |
| 2 | `(7, 8)` | không | - | - | `[5,7,7,8]` |

Đáp án: `i = 1`, nâng, mảng `[5, 7, 7, 8]`, kết quả `True`.

In [38]:
assert trace_greedy([5, 7, 1, 8]) is True
assert descents([5, 7, 1, 8]) == [1]
fixed = [5, 7, 1, 8]
fixed[2] = 7
assert fixed == [5, 7, 7, 8] and is_nondecreasing(fixed)
print("Bài 2 OK")

bắt đầu: [5, 7, 1, 8]
i=0: (5,7) ok            cnt=0 mảng=[5, 7, 1, 8]
i=1: (7,1) GIẢM lần 1 -> nums[0]=5 > 1 nên NÂNG nums[2] = 7 | mảng=[5, 7, 7, 8]
i=2: (7,8) ok            cnt=1 mảng=[5, 7, 7, 8]
không còn cặp giảm thứ hai -> True
Bài 2 OK


### Đáp án Bài 3

In [39]:
def exercise_3_check_solution(nums: List[int]) -> bool:
    nums = nums[:]
    cnt = 0
    for i in range(len(nums) - 1):
        if nums[i] > nums[i + 1]:
            cnt += 1
            if cnt > 1:
                return False
            if i > 0 and nums[i - 1] > nums[i + 1]:
                nums[i + 1] = nums[i]
            else:
                nums[i] = nums[i + 1]
    return True


assert exercise_3_check_solution([4, 2, 3]) is True
assert exercise_3_check_solution([4, 2, 1]) is False
assert exercise_3_check_solution([3, 4, 2, 3]) is False
rng = random.Random(3)
for _ in range(500):
    arr = [rng.randint(-3, 3) for _ in range(rng.randint(1, 8))]
    assert exercise_3_check_solution(arr) == oracle_nd(arr), arr
print("Bài 3 OK")

Bài 3 OK

### Đáp án Bài 4

In [40]:
def exercise_4_min_changes_solution(nums: List[int]) -> int:
    tails = []
    for x in nums:
        p = bisect_right(tails, x)
        if p == len(tails):
            tails.append(x)
        else:
            tails[p] = x
    return len(nums) - len(tails)


assert exercise_4_min_changes_solution([4, 2, 3]) == 1
assert exercise_4_min_changes_solution([4, 2, 1]) == 2
assert exercise_4_min_changes_solution([1, 2, 3]) == 0
assert exercise_4_min_changes_solution([3, 4, 2, 3]) == 2
rng = random.Random(4)
for _ in range(400):
    arr = [rng.randint(-4, 4) for _ in range(rng.randint(1, 9))]
    assert exercise_4_min_changes_solution(arr) == min_changes_dp(arr), arr
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [41]:
def exercise_5_strict_solution(nums: List[int]) -> bool:
    b = [x - i for i, x in enumerate(nums)]
    return exercise_3_check_solution(b)


assert exercise_5_strict_solution([1, 2, 2, 3]) is False
assert exercise_5_strict_solution([1, 3, 3, 4]) is True
assert exercise_5_strict_solution([5, 4, 3]) is False
assert exercise_5_strict_solution([2, 2]) is True
rng = random.Random(5)
for _ in range(400):
    arr = [rng.randint(-3, 5) for _ in range(rng.randint(1, 7))]
    assert exercise_5_strict_solution(arr) == strict_oracle(arr), arr
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
from typing import List

class Solution:
    def checkPossibility(self, nums: List[int]) -> bool:
        cnt = 0
        for i in range(len(nums) - 1):
            if nums[i] > nums[i + 1]:                       # cặp giảm
                cnt += 1
                if cnt > 1:
                    return False
                if i > 0 and nums[i - 1] > nums[i + 1]:     # không hạ trái được
                    nums[i + 1] = nums[i]                   # nâng phải
                else:
                    nums[i] = nums[i + 1]                   # hạ trái
        return True
```

Câu thần chú: **"gặp cặp giảm thì đếm; hàng xóm trái cao hơn `b` thì nâng `b`, không thì hạ `a`; quá một lần là sai."**

### Checklist trước khi submit

- [ ] Chỉ đếm cặp giảm khi `nums[i] > nums[i + 1]` (không dùng `>=`).
- [ ] Có chặn `i > 0` trước khi đọc `nums[i - 1]` (bẫy `nums[-1]`).
- [ ] Xử lý **cả hai** lựa chọn (hạ trái / nâng phải); không chỉ đếm số cặp giảm.
- [ ] Thoát sớm khi `cnt > 1`.
- [ ] Test: `n = 1`, `n = 2`, `[3,4,2,3]` (false), `[3,4,2,5]` (true), `[4,2,3]` (true), toàn bằng nhau.
- [ ] Nhớ rằng cách nộp sửa tại chỗ `nums`; khi tự test hãy truyền bản sao.
- [ ] Skeleton đúng: `class Solution`, method `checkPossibility(self, nums: List[int]) -> bool`; độ phức tạp `O(n)` thời gian, `O(1)` bộ nhớ.